# Xarray-Spatial Corridor: Least-cost corridors, thresholds, and pairwise connectivity

Corridor analysis identifies broad zones of low cumulative travel cost between locations on a friction surface. Where pathfinding tools like A* return a single-cell-wide route, corridors show every cell within a cost budget of the optimal connection. This makes them useful for wildlife connectivity modeling, infrastructure routing, and conservation planning.

### What you'll build

1. Compute a basic corridor between two points on uniform friction
2. Route a corridor through variable friction terrain with a low-cost channel
3. Threshold corridors using absolute and relative cost cutoffs
4. Force corridor detours around NaN barriers
5. Compare precomputed cost-distance surfaces with on-the-fly computation
6. Run pairwise corridors across three source locations

![Corridor analysis preview](images/corridor_analysis_preview.png)

**Jump to a section:**
[Uniform corridor](#Uniform-corridor) | [Variable friction](#Variable-friction) | [Thresholding](#Thresholding) | [Barriers](#Barriers) | [Precomputed surfaces](#Precomputed-surfaces) | [Pairwise corridors](#Pairwise-corridors)

Standard imports plus `cost_distance` and `least_cost_corridor` from xrspatial.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from xrspatial import cost_distance, least_cost_corridor

## Synthetic friction surface

All examples below share a single helper that builds a `DataArray` with y/x coordinates from a numpy array. The friction surfaces themselves are simple enough to construct inline for each section.

In [ ]:
def make_raster(data, res=1.0):
    """Create a DataArray with y/x coordinates from a 2-D array."""
    h, w = data.shape
    return xr.DataArray(
        data.astype(np.float64),
        dims=['y', 'x'],
        coords={'y': np.arange(h) * res, 'x': np.arange(w) * res},
        attrs={'res': (res, res)},
    )

# Quick demo: uniform friction on a 31x31 grid
n = 31
friction = make_raster(np.ones((n, n)))

fig, ax = plt.subplots(figsize=(6, 5))
friction.plot.imshow(ax=ax, cmap='gray', add_colorbar=True,
                     cbar_kwargs={'label': 'Friction cost per cell'})
ax.set_title('Uniform friction surface')
ax.set_axis_off()
plt.tight_layout()

Every cell costs 1.0 to cross. The sections below build different friction patterns on top of this base grid.

## Uniform corridor

With uniform friction, the corridor cost surface is symmetric around the straight line between two sources. `least_cost_corridor` computes cost-distance from each source, sums them, and normalizes so the optimal route has cost 0. The plot below shows this continuous cost surface.

In [ ]:
# Two sources on opposite sides of the grid
src_a_data = np.zeros((n, n))
src_a_data[15, 3] = 1.0

src_b_data = np.zeros((n, n))
src_b_data[15, 27] = 1.0

src_a = make_raster(src_a_data)
src_b = make_raster(src_b_data)

corridor = least_cost_corridor(friction, src_a, src_b)

fig, ax = plt.subplots(figsize=(10, 7.5))
corridor.plot.imshow(ax=ax, cmap='inferno', add_colorbar=True,
                     cbar_kwargs={'label': 'Normalized corridor cost'})
ax.plot(3, 15, 'c^', markersize=12, label='Source A')
ax.plot(27, 15, 'cs', markersize=12, label='Source B')
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()
ax.set_title('Corridor between two points (uniform friction)')
plt.tight_layout()

print(f"Min corridor cost (optimal route): {float(corridor.min()):.4f}")
print(f"Max corridor cost (corners):       {float(corridor.max()):.4f}")

The dark band shows cells on or near the optimal route (cost near 0). Brighter values are farther from the optimal connection in accumulated cost units.

## Variable friction

When friction varies spatially, the corridor bends to follow cheaper terrain. Here a high-friction band (cost 8) blocks the direct path, but a low-cost channel (cost 1) cuts through the middle. The corridor concentrates along the channel because any route outside it must cross the expensive zone.

In [ ]:
# Variable friction: high-cost band with a low-cost channel
friction_var = np.ones((n, n)) * 2.0
friction_var[10:21, :] = 8.0     # high-cost zone
friction_var[14:17, :] = 1.0     # low-cost channel through it

src_a_v = np.zeros((n, n)); src_a_v[15, 2] = 1.0
src_b_v = np.zeros((n, n)); src_b_v[15, 28] = 1.0

friction_v = make_raster(friction_var)
corridor_var = least_cost_corridor(friction_v, make_raster(src_a_v), make_raster(src_b_v))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: friction surface
friction_v.plot.imshow(ax=axes[0], cmap='YlOrBr', add_colorbar=True,
                       cbar_kwargs={'label': 'Friction cost'})
axes[0].set_title('Friction surface')
axes[0].set_axis_off()

# Right: corridor
corridor_var.plot.imshow(ax=axes[1], cmap='inferno', add_colorbar=True,
                         cbar_kwargs={'label': 'Corridor cost'})
axes[1].plot(2, 15, 'c^', markersize=12, label='Source A')
axes[1].plot(28, 15, 'cs', markersize=12, label='Source B')
axes[1].legend(loc='lower right', fontsize=11, framealpha=0.9)
axes[1].set_title('Corridor (routes through channel)')
axes[1].set_axis_off()

plt.tight_layout()

## Thresholding

The `threshold` parameter masks out cells that deviate too far from the optimal route, turning the continuous corridor surface into a discrete zone. Two modes are available:

- **Absolute** (`relative=False`): cells with normalized cost above the threshold are masked.
- **Relative** (`relative=True`): threshold is a fraction of the minimum corridor cost. `threshold=0.10` keeps cells within 10% of the optimal cost.

The plot compares both modes on the uniform friction grid.

In [ ]:
# Absolute threshold
corridor_abs = least_cost_corridor(friction, src_a, src_b, threshold=3.0)

# Relative threshold (10% of minimum corridor cost)
corridor_rel = least_cost_corridor(friction, src_a, src_b,
                                   threshold=0.10, relative=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

corridor_abs.plot.imshow(ax=axes[0], cmap='inferno', add_colorbar=True,
                         cbar_kwargs={'label': 'Corridor cost'})
axes[0].set_title(f'Absolute threshold = 3.0  ({int(np.sum(np.isfinite(corridor_abs.values)))} cells)')
axes[0].set_axis_off()

corridor_rel.plot.imshow(ax=axes[1], cmap='inferno', add_colorbar=True,
                         cbar_kwargs={'label': 'Corridor cost'})
axes[1].set_title(f'Relative threshold = 10%  ({int(np.sum(np.isfinite(corridor_rel.values)))} cells)')
axes[1].set_axis_off()

plt.tight_layout()

Relative thresholds adapt to the scale of the corridor. For long-distance corridors with high total cost, 10% produces a wider zone than for short-distance ones. Absolute thresholds give a fixed-width band in cost units regardless of distance.

<div class="alert alert-block alert-warning">
<b>Threshold units depend on friction units.</b> An absolute threshold of 3.0 means "3.0 accumulated friction cost units above the optimal route." If your friction surface is in seconds-per-meter, the threshold is in seconds. If it is unitless resistance, the threshold is in those same arbitrary units. Double-check what your friction values represent before picking a number.
</div>

## Barriers

NaN cells in the friction surface are impassable. The corridor routes around them, just as `cost_distance` does. Here a vertical wall blocks the direct path, with a single-cell gap at the midpoint. All routes funnel through that gap.

In [ ]:
# Vertical wall with a single-cell gap
friction_barrier = np.ones((n, n))
friction_barrier[5:26, 15] = np.nan
friction_barrier[15, 15] = 1.0   # gap

fric_b = make_raster(friction_barrier)
corridor_barrier = least_cost_corridor(fric_b, src_a, src_b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: friction with barrier visible
barrier_vis = make_raster(np.where(np.isnan(friction_barrier), 0.0, 1.0))
barrier_vis.plot.imshow(ax=axes[0], cmap='gray', add_colorbar=False)
axes[0].legend(handles=[Patch(facecolor='black', label='Barrier (NaN)'),
                        Patch(facecolor='white', label='Passable')],
               loc='lower right', fontsize=11, framealpha=0.9)
axes[0].set_title('Friction surface with barrier')
axes[0].set_axis_off()

# Right: corridor funneled through gap
corridor_barrier.plot.imshow(ax=axes[1], cmap='inferno', add_colorbar=True,
                             cbar_kwargs={'label': 'Corridor cost'})
axes[1].plot(3, 15, 'c^', markersize=12, label='Source A')
axes[1].plot(27, 15, 'cs', markersize=12, label='Source B')
axes[1].legend(loc='lower right', fontsize=11, framealpha=0.9)
axes[1].set_title('Corridor routes through gap')
axes[1].set_axis_off()

plt.tight_layout()

Cells near the gap have low corridor cost. Cells far from it have high cost because all routes must detour through the gap regardless of approach direction.

## Precomputed surfaces

If you already have cost-distance surfaces (e.g., to reuse them across multiple corridor pairs), pass them directly with `precomputed=True`. This skips the internal `cost_distance` calls and goes straight to the corridor computation.

In [ ]:
# Compute cost-distance surfaces once
cd_a = cost_distance(src_a, friction)
cd_b = cost_distance(src_b, friction)

# Use precomputed surfaces for corridor
corridor_pre = least_cost_corridor(friction, cd_a, cd_b, precomputed=True)

# Compare with the regular call
corridor_reg = least_cost_corridor(friction, src_a, src_b)

diff = float(np.nanmax(np.abs(corridor_pre.values - corridor_reg.values)))
print(f"Max difference between precomputed and regular: {diff:.10f}")
print("Results are identical." if diff < 1e-10 else "Results differ!")

## Pairwise corridors

With three or more source locations, pass them as a list with `pairwise=True` to compute corridors for every pair at once. The result is an `xr.Dataset` with one variable per pair. This is faster than calling `least_cost_corridor` in a loop because the cost-distance surfaces are shared where possible.

In [ ]:
# Three habitat patches
positions = [(5, 5), (5, 25), (25, 15)]
sources = []
for r, c in positions:
    s = np.zeros((n, n))
    s[r, c] = 1.0
    sources.append(make_raster(s))

corridors = least_cost_corridor(friction, sources=sources, pairwise=True)

print("Corridor pairs:", list(corridors.data_vars))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name in zip(axes, corridors.data_vars):
    corridors[name].plot.imshow(ax=ax, cmap='inferno', add_colorbar=True,
                                cbar_kwargs={'label': 'Cost'})
    for r, c in positions:
        ax.plot(c, r, 'c*', markersize=14)
    ax.set_title(name, fontsize=11)
    ax.set_axis_off()

axes[0].legend(handles=[Patch(facecolor='cyan', label='Source locations')],
               loc='lower right', fontsize=10, framealpha=0.9)
plt.tight_layout()

# Save preview image
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/corridor_analysis_preview.png', bbox_inches='tight', dpi=120)

<div class="alert alert-block alert-info">
<b>Corridor vs. pathfinding.</b> Use <code>a_star_search</code> when you need a single optimal route (one cell wide). Use <code>least_cost_corridor</code> when you need a zone of low-cost connectivity between two regions. Corridors are more useful for conservation planning because they show the full range of near-optimal routes, not just the single cheapest path.
</div>

### References

- [Least-cost path analysis (Wikipedia)](https://en.wikipedia.org/wiki/Least-cost_path_analysis)
- Beier, P., Majka, D. R., & Spencer, W. D. (2008). [Forks in the Road: Choices in Procedures for Designing Wildland Linkages](https://doi.org/10.1111/j.1523-1739.2008.00942.x). *Conservation Biology*, 22(4), 836-851.
- [xrspatial.pathfinding.least_cost_corridor API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.pathfinding.least_cost_corridor.html)